# Vestibular Schwannoma Data Processing Pipeline
- Inventory extracted imaging files and align accession identifiers with clinical timeline metadata.
- Organize imaging assets into patient- and timepoint-specific folders, copying source files to a normalized workspace.
- Select high-resolution reference scans, derive brain-extracted masks, and run ANTs-based registration across intra- and inter-timepoint studies.
- Generate coregistered outputs, propagate accompanying segmentations, and log failures for later inspection.

In [ ]:
from pathlib import Path
import pandas as pd
import os
# load the meta file that allows for exploration of each subfolder
# dest folder with month-stamped name
root_dir = Path("/standard/gam_ai_group/ExtractedData-08-13-2025")

# read batch CSVs and tag batch_id
dfs = [
    pd.read_csv(root_dir / f"VS_batch{i}" / "Original" / "original_meta.csv").assign(batch_id=i)
    for i in range(1, 6)
]
meta_df = pd.concat(dfs, ignore_index=True)

# concise cleaner: normalize backslashes and strip leading spaces per path segment
col = "Downloaded Original Series Path"
if col not in meta_df.columns:
    raise ValueError(f"Column '{col}' not found")

meta_df[col] = meta_df[col].apply(
    lambda v: v if pd.isna(v) else "/".join(part.lstrip() for part in str(v).replace("\\", "/").split("/"))
)
# save cleaned metadata
meta_df.to_csv("meta_df.csv", index=False)
from pathlib import Path
import pandas as pd

# --- your existing setup code above this remains the same ---

# Column we are cleaning
col = "Downloaded Original Series Path"
if col not in meta_df.columns:
    raise ValueError(f"Column '{col}' not found")

def clean_relpath(v, trim_both=False):
    """Normalize slashes and remove leading spaces for each segment.
    Set trim_both=True to remove both leading & trailing spaces per segment."""
    if pd.isna(v):
        return v
    s = str(v).replace("\\", "/")
    # Choose lstrip (leading only) or strip (both ends)
    strip_fn = (str.strip if trim_both else str.lstrip)
    parts = [strip_fn(seg) for seg in s.split("/") if seg != ""]
    return "/".join(parts)

# Clean the relative path column (leading spaces per level removed)
meta_df[col] = meta_df[col].map(lambda v: clean_relpath(v, trim_both=False))

# --- build abs_path from root_dir / VS_batch{batch_id} / Original / <cleaned relative path> ---
def _build_abs_path(row):
    v = row[col]
    if pd.isna(v) or not str(v).strip():
        return pd.NA
    # ensure treated as relative under "Original"
    rel = str(v).lstrip("/")  # guard against accidental leading slash
    p = root_dir / f"VS_batch{int(row['batch_id'])}" / rel
    return p.as_posix()

meta_df["abs_path"] = meta_df.apply(_build_abs_path, axis=1)

# (Optional) quick validation: no segment should start with a space
# bad_segments = (
#     meta_df[col].astype("string")
#     .str.split("/")
#     .explode()
#     .str.match(r"^\s")
# )
# print("Any segments still starting with a space? ->", bool(bad_segments.fillna(False).any()))

# Save
meta_df.to_csv("meta_df.csv", index=False)

In [2]:
#final.csv has the datapoints and the paths of the files
#[ ] needs image path, segmentation path, Patient_MRI_Days Tracker, segmentation column
#[ ] identify the nifti final of use in the segmentation and image

In [3]:
from pathlib import Path
import pandas as pd

meta_df = pd.read_csv('safe_final_cleaned.csv')

# Create abs_path with proper NaN/None handling
meta_df['abs_path'] = meta_df['Downloaded Original Series Path'].apply(
    lambda x: str(Path(x).resolve()) if pd.notna(x) and isinstance(x, str) else None
)

In [4]:
from pathlib import Path
import pandas as pd
import difflib
import os

def clean_path_components(path_str):
    """Remove leading/trailing spaces from each directory level AND clean path endings."""
    if pd.isna(path_str) or not isinstance(path_str, str):
        return None
    
    # First, strip trailing spaces and periods from the entire path string
    path_str = path_str.rstrip(' .\t')
    
    if not path_str:
        return None
    
    # Split the path into components
    path = Path(path_str)
    parts = list(path.parts)
    
    # Strip leading/trailing whitespace from each component
    cleaned_parts = [part.strip() for part in parts]
    
    # Also strip trailing periods from each component
    cleaned_parts = [part.rstrip('.') for part in cleaned_parts]
    
    # Remove any empty parts
    cleaned_parts = [p for p in cleaned_parts if p]
    
    if len(cleaned_parts) == 0:
        return None
    
    # Handle absolute vs relative paths
    if path.is_absolute():
        cleaned_path = Path(cleaned_parts[0]).joinpath(*cleaned_parts[1:])
    else:
        cleaned_path = Path(*cleaned_parts)
    
    return str(cleaned_path)

def attempt_path_recovery(bad_path_str, similarity_cutoff=0.6):
    """
    Tries to heal a broken path by traversing it from the root.
    If a component doesn't exist, it looks for the closest match in the current directory.
    """
    if not bad_path_str:
        return None

    original_path = Path(bad_path_str)
    parts = list(original_path.parts)
    
    # Start building the path from the first component (drive or root)
    current_path = Path(parts[0])
    
    # If the root itself doesn't exist (e.g. wrong drive letter), we can't do much
    if not current_path.exists():
        return None

    # Iterate through the rest of the components
    for i, part in enumerate(parts[1:], start=1):
        test_path = current_path / part
        
        if test_path.exists():
            # If it exists, keep going
            current_path = test_path
        else:
            # The path broke here. Look inside 'current_path' for a fuzzy match.
            try:
                # List all real items in the current directory
                candidates = [x.name for x in current_path.iterdir()]
                
                # specific check: sometimes case sensitivity is the only issue
                lower_map = {c.lower(): c for c in candidates}
                if part.lower() in lower_map:
                    best_match = lower_map[part.lower()]
                else:
                    # Use difflib to find closest string match
                    matches = difflib.get_close_matches(part, candidates, n=1, cutoff=similarity_cutoff)
                    best_match = matches[0] if matches else None
                
                if best_match:
                    # We found a fix! Update current path with the REAL name
                    current_path = current_path / best_match
                else:
                    # No match found, recovery failed
                    return None
            except (PermissionError, OSError):
                # Can't read directory, stop here
                return None

    return str(current_path)

def process_paths(df, path_col='abs_path', error_csv='errored_paths.csv'):
    """
    1. Checks existence.
    2. Tries to recover missing paths via fuzzy matching.
    3. Returns cleaned DF and final error DF.
    """
    if path_col not in df.columns:
        raise KeyError(f"Column '{path_col}' not found in DataFrame")
    
    # 1. Initial Check
    print("🔍 Performing initial existence check...")
    df['exists'] = df[path_col].apply(lambda p: Path(p).exists() if pd.notna(p) and p else False)
    
    valid_mask = df['exists']
    clean_df = df[valid_mask].copy()
    error_df = df[~valid_mask].copy()
    
    print(f"   Initial Pass: {len(clean_df)} valid, {len(error_df)} missing.")

    # 2. Recovery Phase
    if not error_df.empty:
        print("\n🚑 Attempting to recover broken paths (this may take a moment)...")
        
        # Define a wrapper to print progress or handle errors
        def recover_wrapper(p):
            if pd.isna(p): return None
            return attempt_path_recovery(p)

        # Apply recovery
        error_df['recovered_path'] = error_df[path_col].apply(recover_wrapper)
        
        # Check which ones were successfully recovered
        recovered_mask = error_df['recovered_path'].notna()
        recovered_df = error_df[recovered_mask].copy()
        
        # Update the main path column with the recovered version
        recovered_df[path_col] = recovered_df['recovered_path']
        
        # Drop helper columns
        clean_df = clean_df.drop(columns=['exists'])
        recovered_df = recovered_df.drop(columns=['exists', 'recovered_path'])
        
        # Add recovered rows back to clean_df
        clean_df = pd.concat([clean_df, recovered_df], ignore_index=True)
        
        # Update error_df to only include those that failed recovery
        final_error_df = error_df[~recovered_mask].drop(columns=['exists', 'recovered_path'])
        
        print(f"   ✅ Recovered {len(recovered_df)} paths via fuzzy matching!")
    else:
        final_error_df = pd.DataFrame(columns=df.columns)

    # Save final errors
    if not final_error_df.empty:
        final_error_df.to_csv(error_csv, index=False)
        print(f"⚠️  {len(final_error_df)} paths are unrecoverable - saved to '{error_csv}'")
    
    return clean_df, final_error_df

# --- Main Execution ---

# 1. Load Data
meta_df = pd.read_csv('safe_final_cleaned.csv')

# 2. Basic String Cleaning
print("🧹 Cleaning path strings...")
meta_df['abs_path'] = meta_df['Downloaded Original Series Path'].apply(clean_path_components)

# 3. Process (Check & Recover)
clean_meta_df, error_df = process_paths(meta_df)

# 4. Save
#clean_meta_df.to_csv('cleaned_valid_paths.csv', index=False)
print(f"\n💾 Saved {len(clean_meta_df)} rows with verified paths to 'cleaned_valid_paths.csv'")

# 5. Debugging Output
if not error_df.empty:
    print("\n🔍 Examples of unrecoverable paths:")
    for path in error_df['abs_path'].head(3):
        if pd.notna(path):
            print(f"  Failed: {path}")
            # Show deepest existing parent for debugging
            try:
                p = Path(path)
                while not p.exists() and len(p.parts) > 1:
                    p = p.parent
                print(f"    -> Deepest existing match: {p}")
            except:
                pass

🧹 Cleaning path strings...
🔍 Performing initial existence check...
   Initial Pass: 8560 valid, 23 missing.

🚑 Attempting to recover broken paths (this may take a moment)...
   ✅ Recovered 23 paths via fuzzy matching!

💾 Saved 8583 rows with verified paths to 'cleaned_valid_paths.csv'


In [5]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d')
error_df.to_csv(f'missing_paths_{timestamp}.csv')
# Results in: missing_paths_20260117_143022.csv

In [6]:
# --- Load timeline metadata from the Carina spreadsheet and normalize identifiers ---
# Import the information from deidentified Carina to timeline from database
df_excel = pd.read_excel("VS AI Database, Draft 12.01 Jack Veda reconciled.xlsx")
# Keep only the fields needed to align folder structure with patient/timepoint info
df_time = df_excel[['Carina Accession Number', 'Patient_MRI_Days Tracker']].replace('.', None).dropna(subset=['Patient_MRI_Days Tracker'])
# Retain whitespace/full string for accession number to preserve join key
df_time['Carina Accession Number'] = df_time['Carina Accession Number']
# Split tracker strings into patient ID, scan order, and inter-scan spacing
df_time[['Patient_ID', 'Scan Order', 'Days between scans']] = df_time['Patient_MRI_Days Tracker'].str.split('_', expand=True)
# Ensure scans are in chronological order before merging back to filesystem inventory
df_time = df_time.sort_values(by=['Patient_ID', 'Scan Order'])

In [7]:
meta_db_df = pd.merge(
    df_time,
    clean_meta_df,
    left_on="Carina Accession Number",
    right_on="Accession Number",
    how="inner"
)

# Calculate number of unique patients matched
num_patients_matched = meta_db_df['Patient_ID'].nunique()
print(f"Number of unique patients matched: {num_patients_matched}")

# Find unmatched records from clean_meta_df
unmatched_df = clean_meta_df[
    ~clean_meta_df['Accession Number'].isin(meta_db_df['Accession Number'])
]

# Export unmatched to CSV
#unmatched_df.to_csv('unmatched.csv', index=False)
print(f"Number of unmatched records: {len(unmatched_df)}")
print(f"Unmatched records exported to 'unmatched.csv'")

Number of unique patients matched: 257
Number of unmatched records: 10
Unmatched records exported to 'unmatched.csv'


In [8]:
meta_db_df = meta_db_df.rename(columns={'Labels': 'Series Labels'})

In [9]:
#filter the patients to t2_thin and t1+_thin
print(meta_db_df['Series Labels'].value_counts())

#meta_db_df=meta_db_df[meta_db_df ['Series Labels'].isin(['t2_thin','t1+_thin'])]
meta_db_df=meta_db_df[meta_db_df ['Series Labels'].isin(['t2_thin','t1+_thin','t1+_thick_cor','t1+_thick_ax','t2_thick'])]
print('Filtered Data')
print(meta_db_df['Series Labels'].value_counts())

Series Labels
t2_thin          549
t1+_thick_cor    410
t1+_thick_ax     404
t1+_thin         292
t2_thick         155
t1_pre            67
flair             10
Name: count, dtype: int64
Filtered Data
Series Labels
t2_thin          549
t1+_thick_cor    410
t1+_thick_ax     404
t1+_thin         292
t2_thick         155
Name: count, dtype: int64


In [10]:
import pandas as pd
import os
import shutil
from pathlib import Path
import re
from tqdm import tqdm

def find_nifti_files(abs_path):
    """
    Find NIfTI files in the directory.
    Returns: (image_path, seg_path, multiple_segs_flag, all_seg_files)
    """
    if not os.path.exists(abs_path):
        return None, None, False, []
    
    # Get all .nii.gz files
    nii_files = [f for f in os.listdir(abs_path) if f.endswith('.nii.gz')]
    
    # Find image file (no 'uvauser' in name)
    image_files = [f for f in nii_files if 'uvauser' not in f.lower()]
    image_files = [f for f in nii_files if 'image' in f.lower()]
    image_path = os.path.join(abs_path, image_files[0]) if image_files else None
    
    # Find segmentation files (with 'uvauser' in name)
    seg_files = [f for f in nii_files if 'uvauser' in f.lower()]
    
    # Check for multiple segmentations
    multiple_segs = len(seg_files) > 1
    
    # Select segmentation: prefer one with 'copy' if multiple exist
    seg_path = None
    if seg_files:
        if multiple_segs:
            # Try to find one with 'copy' in the name
            copy_segs = [f for f in seg_files if 'copy' in f.lower()]
            seg_path = os.path.join(abs_path, copy_segs[0]) if copy_segs else os.path.join(abs_path, seg_files[0])
        else:
            seg_path = os.path.join(abs_path, seg_files[0])
    
    return image_path, seg_path, multiple_segs, seg_files

def copy_to_organized_folder(src_path, base_output_dir, series_label, patient_mri_days):
    """
    Copy file to organized folder structure with prefixed filename.
    Prefix format: Patient_MRI_Days_Tracker_SeriesLabels_original_filename
    Returns the new path
    """
    if not src_path or not os.path.exists(src_path):
        return None
    
    # Create destination directory
    dest_dir = os.path.join(base_output_dir, str(series_label), str(patient_mri_days))
    os.makedirs(dest_dir, exist_ok=True)
    
    # Get original filename
    original_filename = os.path.basename(src_path)
    
    # Create prefixed filename: Patient_MRI_Days_Tracker_SeriesLabels_original_filename
    prefixed_filename = f"{patient_mri_days}_{series_label}_{original_filename}"
    
    # Create destination path with prefixed filename
    dest_path = os.path.join(dest_dir, prefixed_filename)
    
    # Copy file
    shutil.copy2(src_path, dest_path)
    
    return dest_path

def process_meta_db(meta_db_df, base_output_dir):
    """
    Process the meta database dataframe to find and organize NIfTI files.
    
    Parameters:
    - meta_db_df: DataFrame with metadata
    - base_output_dir: Base directory for organized output
    
    Returns:
    - Updated DataFrame with new columns
    """
    # Create new columns
    meta_db_df['original_image_path'] = None
    meta_db_df['original_seg_path'] = None
    meta_db_df['multiple_segs_flag'] = False
    meta_db_df['all_seg_filenames'] = None  # New column for all seg filenames
    meta_db_df['modality_image_path'] = None
    meta_db_df['modality_seg_path'] = None
    
    # Track multiple segmentations
    multiple_seg_rows = []
    
    # Iterate through each row with progress bar
    for idx, row in tqdm(meta_db_df.iterrows(), total=len(meta_db_df), desc="Processing NIfTI files"):
        abs_path = row['abs_path']
        series_label = row['Series Labels']
        patient_mri_days = row['Patient_MRI_Days Tracker']
        
        # Find NIfTI files
        image_path, seg_path, multiple_segs, all_seg_files = find_nifti_files(abs_path)
        
        # Update original paths
        meta_db_df.at[idx, 'original_image_path'] = image_path
        meta_db_df.at[idx, 'original_seg_path'] = seg_path
        meta_db_df.at[idx, 'multiple_segs_flag'] = multiple_segs
        
        # Store all segmentation filenames as a list (only if there are multiple)
        if multiple_segs:
            meta_db_df.at[idx, 'all_seg_filenames'] = all_seg_files
        
        # Copy files to organized structure with prefixed filenames
        if image_path:
            new_image_path = copy_to_organized_folder(
                image_path, base_output_dir, series_label, patient_mri_days
            )
            meta_db_df.at[idx, 'modality_image_path'] = new_image_path
        
        if seg_path:
            new_seg_path = copy_to_organized_folder(
                seg_path, base_output_dir, series_label, patient_mri_days
            )
            meta_db_df.at[idx, 'modality_seg_path'] = new_seg_path
        
        if multiple_segs:
            multiple_seg_rows.append(idx)
    
    # Print summary of multiple segmentations
    if multiple_seg_rows:
        print(f"\n⚠️  Found {len(multiple_seg_rows)} rows with multiple segmentations:")
        print(f"   Row indices: {multiple_seg_rows}")
        print("\n   Details:")
        for idx in multiple_seg_rows:
            seg_list = meta_db_df.at[idx, 'all_seg_filenames']
            print(f"   Row {idx}: {len(seg_list)} segmentations - {seg_list}")
    
    return meta_db_df

# Usage in Jupyter Notebook:
# Uncomment and run the following lines after loading your dataframe

# Set base output directory
base_output_dir = './raw_studies'

# Process the dataframe
updated_meta_db_df = process_meta_db(meta_db_df, base_output_dir)

# Save the updated dataframe
#updated_meta_db_df.to_csv('updated_metadata.csv', index=False)

# Display results
updated_meta_db_df[['Patient_MRI_Days Tracker', 'Series Labels', 'original_image_path', 
                      'original_seg_path', 'multiple_segs_flag', 'all_seg_filenames',
                      'modality_image_path', 'modality_seg_path']].head()

Processing NIfTI files: 100%|██████████| 1810/1810 [09:10<00:00,  3.29it/s]



⚠️  Found 307 rows with multiple segmentations:
   Row indices: [82, 87, 116, 129, 141, 180, 182, 202, 207, 211, 233, 257, 283, 287, 318, 342, 358, 359, 374, 404, 424, 426, 429, 452, 461, 467, 478, 545, 551, 557, 605, 638, 652, 654, 656, 660, 673, 695, 706, 708, 814, 818, 819, 842, 865, 867, 870, 872, 899, 914, 934, 955, 1026, 1048, 1057, 1083, 1089, 1095, 1243, 1257, 1318, 1320, 1330, 1332, 1350, 1364, 1631, 1652, 1658, 1666, 1706, 1708, 1715, 1719, 1720, 1891, 1907, 1928, 1936, 1942, 2030, 2044, 2056, 2152, 2182, 2343, 2353, 2383, 2397, 2448, 2453, 2454, 2497, 2500, 2502, 2512, 2514, 2519, 2570, 2576, 2605, 2630, 2633, 2749, 2795, 2962, 2982, 3175, 3178, 3180, 3184, 3212, 3233, 3245, 3250, 3255, 3259, 3263, 3330, 3335, 3415, 3418, 3421, 3422, 3426, 3488, 3506, 3507, 3509, 3583, 3602, 3626, 3698, 3699, 3700, 3749, 3752, 3753, 3761, 3762, 3766, 3786, 3800, 3802, 3808, 3919, 3932, 3957, 3959, 3960, 3964, 3965, 3976, 4044, 4092, 4153, 4189, 4190, 4196, 4199, 4202, 4220, 4237, 4453, 4479

,Patient_MRI_Days Tracker,Series Labels,original_image_path,original_seg_path,multiple_segs_flag,all_seg_filenames,modality_image_path,modality_seg_path
39,116_1_290,t2_thick,Original/VS_batch4_0183 VS_batch4_0183/MRI BRA...,None,False,None,./raw_studies/t2_thick/116_1_290/116_1_290_t2_...,None
63,126_0_0,t2_thick,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,None,./raw_studies/t2_thick/126_0_0/126_0_0_t2_thic...,./raw_studies/t2_thick/126_0_0/126_0_0_t2_thic...
66,126_0_0,t1+_thick_cor,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,None,./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1...,./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1...
70,126_0_0,t1+_thick_ax,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,None,./raw_studies/t1+_thick_ax/126_0_0/126_0_0_t1+...,./raw_studies/t1+_thick_ax/126_0_0/126_0_0_t1+...
77,126_1_312,t1+_thick_ax,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,False,None,./raw_studies/t1+_thick_ax/126_1_312/126_1_312...,./raw_studies/t1+_thick_ax/126_1_312/126_1_312...


In [11]:
updated_meta_db_df

,Carina Accession Number,Patient_MRI_Days Tracker,Patient_ID,Scan Order,Days between scans,Patient ID,Patient Name,Patient Birth Date,Study Date,Accession Number,...,Series Labels,Notes,Study_Labels,abs_path,original_image_path,original_seg_path,multiple_segs_flag,all_seg_filenames,modality_image_path,modality_seg_path
39,VS_batch4_0183_002,116_1_290,116,1,290,VS_batch4_0183,VS_batch4_0183,11/23/1949,11/19/2017,VS_batch4_0183_002,...,t2_thick,NaN,non_contrast,Original/VS_batch4_0183 VS_batch4_0183/MRI BRA...,Original/VS_batch4_0183 VS_batch4_0183/MRI BRA...,None,False,None,./raw_studies/t2_thick/116_1_290/116_1_290_t2_...,None
63,VS_batch2_0001_001,126_0_0,126,0,0,VS_batch2_0001,VS_batch2_0001,5/19/1976,10/12/2001,VS_batch2_0001_001,...,t2_thick,NaN,qc1,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,None,./raw_studies/t2_thick/126_0_0/126_0_0_t2_thic...,./raw_studies/t2_thick/126_0_0/126_0_0_t2_thic...
66,VS_batch2_0001_001,126_0_0,126,0,0,VS_batch2_0001,VS_batch2_0001,5/19/1976,10/12/2001,VS_batch2_0001_001,...,t1+_thick_cor,NaN,qc1,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,None,./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1...,./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1...
70,VS_batch2_0001_001,126_0_0,126,0,0,VS_batch2_0001,VS_batch2_0001,5/19/1976,10/12/2001,VS_batch2_0001_001,...,t1+_thick_ax,NaN,qc1,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,None,./raw_studies/t1+_thick_ax/126_0_0/126_0_0_t1+...,./raw_studies/t1+_thick_ax/126_0_0/126_0_0_t1+...
77,VS_batch1_0001_001,126_1_312,126,1,312,VS_batch1_0001,VS_batch1_0001,4/14/1983,7/15/2009,VS_batch1_0001_001,...,t1+_thick_ax,NaN,qc1,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,False,None,./raw_studies/t1+_thick_ax/126_1_312/126_1_312...,./raw_studies/t1+_thick_ax/126_1_312/126_1_312...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8552,VS_batch1_0089_002,95_1_182,95,1,182,VS_batch1_0089,VS_batch1_0089,4/20/1930,1/21/1982,VS_batch1_0089_002,...,t1+_thick_ax,NaN,qc1,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,False,None,./raw_studies/t1+_thick_ax/95_1_182/95_1_182_t...,./raw_studies/t1+_thick_ax/95_1_182/95_1_182_t...
8558,VS_batch1_0089_002,95_1_182,95,1,182,VS_batch1_0089,VS_batch1_0089,4/20/1930,1/21/1982,VS_batch1_0089_002,...,t2_thin,NaN,qc1,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,True,"[RVS _uvauser2-copy.nii.gz, RVS _uvauser2.nii.gz]",./raw_studies/t2_thin/95_1_182/95_1_182_t2_thi...,./raw_studies/t2_thin/95_1_182/95_1_182_t2_thi...
8562,VS_batch1_0089_003,95_2_574,95,2,574,VS_batch1_0089,VS_batch1_0089,4/20/1930,2/17/1983,VS_batch1_0089_003,...,t2_thin,NaN,qc1,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,True,"[RVS _uvauser2-copy.nii.gz, RVS _uvauser2.nii.gz]",./raw_studies/t2_thin/95_2_574/95_2_574_t2_thi...,./raw_studies/t2_thin/95_2_574/95_2_574_t2_thi...
8569,VS_batch1_0089_003,95_2_574,95,2,574,VS_batch1_0089,VS_batch1_0089,4/20/1930,2/17/1983,VS_batch1_0089_003,...,t1+_thick_cor,NaN,qc1,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,False,None,./raw_studies/t1+_thick_cor/95_2_574/95_2_574_...,./raw_studies/t1+_thick_cor/95_2_574/95_2_574_...


In [12]:
updated_meta_db_df= updated_meta_db_df.drop(['Patient Name',
       'Patient Birth Date', 'Study Date', 'Accession Number',
       'Downloaded Original Series Path','Notes',],axis=1)


In [13]:
updated_meta_db_df = updated_meta_db_df.rename(columns={
    "Series Labels": "Study Type",
    "modality_image_path":"image path",
    "modality_seg_path": "segmentation path"

})
print(updated_meta_db_df.Study_Labels.unique())

['non_contrast' 'qc1' 'check' 'no_thins']


In [14]:
#check if there is anything of concern in the study labels
updated_meta_db_df=updated_meta_db_df[~updated_meta_db_df['Study_Labels'].isin(['not_vs','non_contrast'])]

In [15]:
from datetime import datetime

# Generate filename with current date
date_str = datetime.now().strftime("%Y-%m-%d")
filename = f"meta_df_{date_str}.csv"

# Save DataFrame to CSV
updated_meta_db_df.to_csv(filename, index=False)

print(f"DataFrame saved to {filename}")

DataFrame saved to meta_df_2026-02-10.csv


In [16]:
# ------------------------------------------------------------
# 1) Configuration & imports
# ------------------------------------------------------------
# Set a reproducible seed (used later if randomness added) and number of threads for ITK/ANTs operations.
# Increase THREADS to leverage more CPU cores for registration & preprocessing.
SEED = 42
THREADS = 16

import os
# Set ITK thread limit so ANTs/SimpleITK honors the chosen concurrency.
os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = str(THREADS)
print(f"Set ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS to {THREADS}.")

import gc
import shutil
import subprocess
import shlex
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import nibabel as nib
import tifffile
import SimpleITK as sitk
import ants
from pathlib import Path
import numpy as np

# Load the metadata CSV then remove undesired study types and rows missing image paths.
# The regex excludes 'T1 pre' and 'T2 Thick' variants (case-insensitive).

meta_df = pd.read_csv(filename)
meta_df = meta_df[~meta_df['Study Type'].str.contains(r'T1 pre|T2 Thick', case=False, na=False)].dropna(subset = 'image path')
# List of candidate T2 thin images (example subset potentially targeted for HD-BET).
images = meta_df[meta_df['Study Type']=='t2_thin']['image path'].dropna() 
print(len(images))

Set ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS to 16.
542


In [17]:
meta_df

,Carina Accession Number,Patient_MRI_Days Tracker,Patient_ID,Scan Order,Days between scans,Patient ID,Series Instance UID,Series Index,Series Description,Study Type,Study_Labels,abs_path,original_image_path,original_seg_path,multiple_segs_flag,all_seg_filenames,image path,segmentation path
0,VS_batch2_0001_001,126_0_0,126,0,0,VS_batch2_0001,1.2.826.0.1.3680043.8.498.11849212733585833999...,1,t2_propeller_ax,t2_thick,qc1,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,NaN,./raw_studies/t2_thick/126_0_0/126_0_0_t2_thic...,./raw_studies/t2_thick/126_0_0/126_0_0_t2_thic...
1,VS_batch2_0001_001,126_0_0,126,0,0,VS_batch2_0001,1.2.826.0.1.3680043.8.498.93570814980014534584...,4,t1_pg_cor,t1+_thick_cor,qc1,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,NaN,./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1...,./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1...
2,VS_batch2_0001_001,126_0_0,126,0,0,VS_batch2_0001,1.2.826.0.1.3680043.8.498.37810492909716881335...,8,t1_pg_ax,t1+_thick_ax,qc1,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,Original/VS_batch2_0001 VS_batch2_0001/MR NO R...,False,NaN,./raw_studies/t1+_thick_ax/126_0_0/126_0_0_t1+...,./raw_studies/t1+_thick_ax/126_0_0/126_0_0_t1+...
3,VS_batch1_0001_001,126_1_312,126,1,312,VS_batch1_0001,1.2.826.0.1.3680043.8.498.28817049015689088667...,7,NaN,t1+_thick_ax,qc1,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,False,NaN,./raw_studies/t1+_thick_ax/126_1_312/126_1_312...,./raw_studies/t1+_thick_ax/126_1_312/126_1_312...
4,VS_batch1_0001_001,126_1_312,126,1,312,VS_batch1_0001,1.2.826.0.1.3680043.8.498.13013546638123012911...,8,NaN,t1+_thick_cor,qc1,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,Original/VS_batch1_0001 VS_batch1_0001/MRI TEM...,False,NaN,./raw_studies/t1+_thick_cor/126_1_312/126_1_31...,./raw_studies/t1+_thick_cor/126_1_312/126_1_31...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,VS_batch1_0089_002,95_1_182,95,1,182,VS_batch1_0089,1.2.826.0.1.3680043.8.498.10208435538285008774...,5,NaN,t1+_thick_ax,qc1,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,False,NaN,./raw_studies/t1+_thick_ax/95_1_182/95_1_182_t...,./raw_studies/t1+_thick_ax/95_1_182/95_1_182_t...
1802,VS_batch1_0089_002,95_1_182,95,1,182,VS_batch1_0089,1.2.826.0.1.3680043.8.498.14209899306722623411...,11,NaN,t2_thin,qc1,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,Original/VS_batch1_0089 VS_batch1_0089/BRAIN W...,True,"['RVS _uvauser2-copy.nii.gz', 'RVS _uvauser2.n...",./raw_studies/t2_thin/95_1_182/95_1_182_t2_thi...,./raw_studies/t2_thin/95_1_182/95_1_182_t2_thi...
1803,VS_batch1_0089_003,95_2_574,95,2,574,VS_batch1_0089,1.2.826.0.1.3680043.8.498.57214864933569825812...,1,NaN,t2_thin,qc1,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,True,"['RVS _uvauser2-copy.nii.gz', 'RVS _uvauser2.n...",./raw_studies/t2_thin/95_2_574/95_2_574_t2_thi...,./raw_studies/t2_thin/95_2_574/95_2_574_t2_thi...
1804,VS_batch1_0089_003,95_2_574,95,2,574,VS_batch1_0089,1.2.826.0.1.3680043.8.498.80100859856888426958...,8,NaN,t1+_thick_cor,qc1,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,Original/VS_batch1_0089 VS_batch1_0089/MR NO R...,False,NaN,./raw_studies/t1+_thick_cor/95_2_574/95_2_574_...,./raw_studies/t1+_thick_cor/95_2_574/95_2_574_...


In [18]:
# === HD-BET processing pipeline ===
# Builds a working dataframe of candidate images, stages unprocessed ones into hd_input,
# executes HD-BET (batch or per-file fallback), collects outputs, writes manifests, then cleans temp inputs.
# Assumptions: meta_df has columns 'image path' and 'Study Type'.


# --------------------------
# Config (edit as needed)
# --------------------------
cwd = os.getcwd()  # or set to a specific working directory path string
raw_path = os.path.join(cwd, 'hd_input')   # input folder for HD-BET
bet_path = os.path.join(cwd, 'hd_output')  # output folder for HD-BET

# Filter which rows to include from meta_df
FILTER_STUDY_TYPE = None  # set to None to use all rows

# meta_df columns
PATH_COLUMN = "image path"      # path to .nii or .nii.gz
STUDY_COL   = "Study Type"

# HD-BET binary path (uses this if it exists; else falls back to PATH)
hd_bet_path = '/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet'

# HD-BET flags
DEVICE = "cuda"
DISABLE_TTA = True                   # if your build doesn't support it, change to: ["--tta", "0"]
SAVE_ONLY_MASK = True                # True -> --save_bet_mask --no_bet_image

# What counts as "already processed" in hd_output:
KNOWN_OUTPUT_SUFFIXES = ['_bet', '_bet_mask', '_mask']

# Whether to wipe hd_input before copying (recommended for clean runs)
CLEAN_HD_INPUT_FIRST = True

# --------------------------
# Helpers
# --------------------------
def _strip_nii_ext(fn: str) -> str:
    """Return filename without the .nii or .nii.gz extension."""
    if fn.endswith('.nii.gz'):
        return fn[:-7]
    if fn.endswith('.nii'):
        return fn[:-4]
    return fn

def _expected_outputs(basename_noext: str, suffixes=None):
    """Build set of possible HD-BET output names (with .nii.gz) given a base."""
    suffixes = suffixes or ['_bet', '_bet_mask', '_mask']
    return {f"{basename_noext}{s}.nii.gz" for s in suffixes}

# --------------------------
# 1) Build df from meta_df
# --------------------------
if FILTER_STUDY_TYPE is None:
    sel = meta_df.copy()
else:
    if STUDY_COL not in meta_df.columns:
        raise KeyError(f"'{STUDY_COL}' not found in meta_df.")
    sel = meta_df[meta_df[STUDY_COL] == FILTER_STUDY_TYPE].copy()

if PATH_COLUMN not in sel.columns:
    raise KeyError(f"'{PATH_COLUMN}' not found in meta_df.")

# Drop rows with missing paths, and create (Image_Path, Image)
sel = sel[sel[PATH_COLUMN].notna()].copy()
sel["Image_Path"] = sel[PATH_COLUMN].astype(str)
sel["Image"] = sel["Image_Path"].apply(os.path.basename)

# Keep only what we need for processing, but retain Study Type for context
keep_cols = [c for c in ["Image", "Image_Path", STUDY_COL] if c in sel.columns]
df = sel[keep_cols].copy()

print(f"[INFO] Candidate rows from meta_df: {len(df)}" + (f" | Study Type == '{FILTER_STUDY_TYPE}'" if FILTER_STUDY_TYPE else ""))

# ------------------------------------------
# 2) Prepare HD-BET input/output folders
# ------------------------------------------
os.makedirs(raw_path, exist_ok=True)
os.makedirs(bet_path, exist_ok=True)

# If desired, clear hd_input so this run is fresh
if CLEAN_HD_INPUT_FIRST:
    for p in Path(raw_path).glob("*"):
        if p.is_file():
            p.unlink()

# List of existing outputs (if hd_output already exists)
existing_files = set(os.listdir(bet_path)) if os.path.exists(bet_path) else set()


[INFO] Candidate rows from meta_df: 1793


In [19]:
# ----------------------------------------------------
# 3) Decide: TODO vs SKIPPED; copy only unprocessed
# ----------------------------------------------------
rows_todo = []
rows_skipped = []

print(f"\n=== Processing {len(df)} total files ===")
print(f"Found {len(existing_files)} existing output files in {bet_path}")

for idx, row in df.iterrows():
    img_name = str(row["Image"])
    img_path = str(row["Image_Path"])
    
    # NIfTI guard
    if not (img_name.endswith(".nii") or img_name.endswith(".nii.gz")):
        rs = row.to_dict()
        rs["Reason"] = "unsupported_extension"
        rows_skipped.append(rs)
        print(f"❌ Skipped {img_name}: unsupported extension")
        continue
    
    # Already processed?
    base_noext = _strip_nii_ext(img_name)
    expected = _expected_outputs(base_noext, KNOWN_OUTPUT_SUFFIXES)
    matches = expected & existing_files
    
    if matches:
        rs = row.to_dict()
        rs["Reason"] = "already_processed"
        rs["ExistingOutputsMatched"] = ";".join(sorted(matches))
        rows_skipped.append(rs)
        print(f"⏭️  Skipped {img_name}: already processed (found {len(matches)} outputs)")
        continue
    
    # Missing file?
    if not img_path or not os.path.exists(img_path):
        rs = row.to_dict()
        rs["Reason"] = "missing_file"
        rows_skipped.append(rs)
        print(f"❌ Skipped {img_name}: file not found at {img_path}")
        continue
    
    # Copy to hd_input
    dest = os.path.join(raw_path, img_name)
    shutil.copy(img_path, dest)
    rt = row.to_dict()
    rt["CopiedTo"] = dest
    rows_todo.append(rt)
    print(f"✅ Copied {img_name} to hd_input")

todo_df = pd.DataFrame(rows_todo)
skipped_df = pd.DataFrame(rows_skipped)

print(f"\n=== Summary ===")
print(f"To process: {len(todo_df)}")
print(f"Skipped: {len(skipped_df)}")
if len(skipped_df) > 0:
    print(f"\nSkip reasons:")
    print(skipped_df["Reason"].value_counts())


=== Processing 1793 total files ===
Found 1645 existing output files in /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_output
⏭️  Skipped 126_0_0_t2_thick_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_0_0_t1+_thick_cor_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_0_0_t1+_thick_ax_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_1_312_t1+_thick_ax_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_1_312_t1+_thick_cor_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_1_312_t2_thin_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_1_312_t1+_thick_cor_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_2_582_t1+_thick_cor_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_2_582_t2_thin_image.nii.gz: already processed (found 1 outputs)
⏭️  Skipped 126_2_582_t1+_thick_ax_image.nii.gz: already processed (found 1 outputs)
⏭️  Skip

In [20]:
# ----------------------------------------
# 4) Write CSV manifests (todo & skipped)
# ----------------------------------------
timestamp = datetime.now().strftime("%Y%m")
todo_csv = os.path.join(cwd, f"hd_bet_todo_{timestamp}.csv")
skipped_csv = os.path.join(cwd, f"hd_bet_skipped_{timestamp}.csv")

todo_df.to_csv(todo_csv, index=False)
skipped_df.to_csv(skipped_csv, index=False)
print(f"[INFO] Wrote: {todo_csv}")
print(f"[INFO] Wrote: {skipped_csv}")


[INFO] Wrote: /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_bet_todo_202602.csv
[INFO] Wrote: /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_bet_skipped_202602.csv


In [21]:

# ----------------------------------------
# 5) Run HD-BET (if anything to process)
# ----------------------------------------
if todo_df.empty:
    print("[INFO] Nothing to run. All images are either processed, missing, or unsupported.")
    print(f"[SUMMARY] skipped={len(skipped_df)}, todo=0")
else:
    # Resolve hd-bet command
    if hd_bet_path and Path(hd_bet_path).exists():
        hd_bet_cmd = hd_bet_path
    else:
        # fallback to PATH lookup
        hd_bet_cmd = shutil.which("hd-bet")
        if not hd_bet_cmd:
            raise FileNotFoundError(
                "hd-bet command not found. Set hd_bet_path to the full path or ensure it's on PATH."
            )

    args = [
        shlex.quote(hd_bet_cmd),
        "-i", shlex.quote(raw_path),
        "-o", shlex.quote(bet_path),
        "-device", shlex.quote(DEVICE),
    ]
    if DISABLE_TTA:
        # If your build doesn't support this flag, replace with: args.extend(["--tta", "0"])
        args.append("--disable_tta")
    if SAVE_ONLY_MASK:
        args.extend(["--save_bet_mask", "--no_bet_image"])

    command = " ".join(args)
    print(f"[INFO] Running HD-BET on {len(todo_df)} files (batch):\n{command}")

    # Use subprocess for better error reporting. On batch failure, retry per-file with explicit output path.
    try:
        subprocess.run(command, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] HD-BET batch run failed with return code {e.returncode}. Falling back to per-file runs.")
        per_file_failures = []
        # iterate over files actually copied into raw_path (use todo_df to preserve original ordering/info)
        for rt in rows_todo:
            input_fp = rt["CopiedTo"]
            if not os.path.exists(input_fp):
                print(f"[WARN] Input missing for per-file retry: {input_fp}")
                per_file_failures.append((input_fp, "missing_input"))
                continue

            base_noext = _strip_nii_ext(os.path.basename(input_fp))
            # direct full output file path ending in .nii.gz inside hd_output
            out_fp = os.path.join(bet_path, f"{base_noext}.nii.gz")

            per_args = [
                shlex.quote(hd_bet_cmd),
                "-i", shlex.quote(input_fp),
                "-o", shlex.quote(out_fp),
                "-device", shlex.quote(DEVICE),
            ]
            if DISABLE_TTA:
                per_args.append("--disable_tta")
            if SAVE_ONLY_MASK:
                per_args.extend(["--save_bet_mask", "--no_bet_image"])

            per_command = " ".join(per_args)
            print(f"[INFO] Running HD-BET for single file:\n{per_command}")
            try:
                subprocess.run(per_command, shell=True, check=True)
            except subprocess.CalledProcessError as e2:
                print(f"[ERROR] HD-BET failed for {input_fp} with return code {e2.returncode}")
                per_file_failures.append((input_fp, e2.returncode))
            except Exception as ex:
                print(f"[ERROR] Unexpected error for {input_fp}: {ex}")
                per_file_failures.append((input_fp, str(ex)))
        if per_file_failures:
            print(f"[WARN] Per-file retries had {len(per_file_failures)} failures. See logs above.")
    finally:
        # Clean up input folder and free memory
        try:
            shutil.rmtree(raw_path)
            print(f"[INFO] Removed temp folder: {raw_path}")
        except Exception as e:
            print(f"[WARN] Could not remove {raw_path}: {e}")
        gc.collect()

    print(f"[SUMMARY] processed={len(todo_df)}, skipped={len(skipped_df)}")
try:
    shutil.rmtree(raw_path)
    print(f"[INFO] Removed temp folder: {raw_path}")
except Exception as e:
    print(f"[WARN] Could not remove {raw_path}: {e}")
gc.collect()

[INFO] Running HD-BET on 57 files (batch):
/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet -i '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input' -o '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_output' -device cuda --disable_tta --save_bet_mask --no_bet_image

########################
If you are using hd-bet, please cite the following papers:

Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.

Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
########################

There are 55 cases in the source folder
I am processing 0 out of 1 (max 

Process SpawnProcess-11:
Traceback (most recent call last):
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunetv2/inference/data_iterators.py", line 58, in preprocess_fromfiles_save_to_queue
    raise e
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunetv2/inference/data_iterators.py", line 31, in preprocess_fromfiles_save_to_queue
    data, seg, data_properties = preprocessor.run_case(list_of_lists[idx],
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunetv2/preprocessing/preprocessors/default_preprocessor.py", line 146, in run_case

[ERROR] HD-BET batch run failed with return code 1. Falling back to per-file runs.
[INFO] Running HD-BET for single file:
/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet -i '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input/16_1_1850_t2_thick_image.nii.gz' -o '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_output/16_1_1850_t2_thick_image.nii.gz' -device cuda --disable_tta --save_bet_mask --no_bet_image

########################
If you are using hd-bet, please cite the following papers:

Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.

Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image s

Traceback (most recent call last):
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet", line 7, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/HD_BET/entry_point.py", line 53, in main
    hdbet_predict(args.input,args.output, predictor, keep_brain_mask=args.save_bet_mask,
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/HD_BET/hd_bet_prediction.py", line 69, in hdbet_predict
    predictor.predict_from_files(
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunetv2/inference/predict_from_raw_data.py", line 268, in predict_from_files
    return self.predict_from_data_iterator(data_iterator, save_probabilities, num_processes_segmentation_export)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunet

[ERROR] HD-BET failed for /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input/16_1_1850_t2_thick_image.nii.gz with return code 1
[INFO] Running HD-BET for single file:
/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet -i '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input/175_0_0_t2_thick_image.nii.gz' -o '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_output/175_0_0_t2_thick_image.nii.gz' -device cuda --disable_tta --save_bet_mask --no_bet_image

########################
If you are using hd-bet, please cite the following papers:

Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.

Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU

...

Traceback (most recent call last):
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet", line 7, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/HD_BET/entry_point.py", line 53, in main
    hdbet_predict(args.input,args.output, predictor, keep_brain_mask=args.save_bet_mask,
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/HD_BET/hd_bet_prediction.py", line 69, in hdbet_predict
    predictor.predict_from_files(
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunetv2/inference/predict_from_raw_data.py", line 268, in predict_from_files
    return self.predict_from_data_iterator(data_iterator, save_probabilities, num_processes_segmentation_export)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunet

[ERROR] HD-BET failed for /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input/788_0_0_t2_thick_image.nii.gz with return code 1
[INFO] Running HD-BET for single file:
/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet -i '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input/84_1_851_t2_thin_image.nii.gz' -o '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_output/84_1_851_t2_thin_image.nii.gz' -device cuda --disable_tta --save_bet_mask --no_bet_image

########################
If you are using hd-bet, please cite the following papers:

Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificial neural networks. arXiv preprint arXiv:1901.11341, 2019.

Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-N

Traceback (most recent call last):
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/bin/hd-bet", line 7, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/HD_BET/entry_point.py", line 53, in main
    hdbet_predict(args.input,args.output, predictor, keep_brain_mask=args.save_bet_mask,
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/HD_BET/hd_bet_prediction.py", line 69, in hdbet_predict
    predictor.predict_from_files(
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunetv2/inference/predict_from_raw_data.py", line 268, in predict_from_files
    return self.predict_from_data_iterator(data_iterator, save_probabilities, num_processes_segmentation_export)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mnn7bx/.conda/envs/nguyen_ants_env/lib/python3.11/site-packages/nnunet

[ERROR] HD-BET failed for /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input/84_1_851_t2_thin_image.nii.gz with return code 1
[WARN] Per-file retries had 57 failures. See logs above.
[INFO] Removed temp folder: /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input
[SUMMARY] processed=57, skipped=1736
[WARN] Could not remove /sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input: [Errno 2] No such file or directory: '/sfs/ceph/standard/gam_ai_group/600 qc1 scans/studies_20260130105647718474/hd_input'


0

In [22]:
# === Entropy, voxel volume, file size, and segmentation volume computation ===
# For every row in meta_df, derive voxel dimensions (from row columns or file headers) and compute:
#   - voxel_volume = product of (dx, dy, dz) when available
#   - voxel_unit = unit of measure for voxel dimensions (e.g., 'mm', 'um', 'pixels')
#   - volume_unit = unit of measure for volumes (e.g., 'mm³', 'um³', 'pixels³')
#   - entropy (Shannon, base 2) over intensities using histogram (approximate if large).
#   - file_size_mb = file size in megabytes
#   - coregistered_image = path to coregistered image (replaces 'raw_studies' with COREGISTERED_FOLDER)
#   - coregistered_segmentation = path to coregistered segmentation (replaces 'raw_studies' with COREGISTERED_FOLDER)
#   - segmentation_volume = volume of segmentation (non-zero voxels * voxel_volume) in cubic units
#   - antspy_spacing = spacing tuple extracted directly from ANTsPy image
# Results added to meta_df and cached in meta_df.csv for reuse.

# Optional runtime toggles: set to False if you want exact (but slower) entropy
APPROXIMATE_ENTROPY = True
MAX_ENTROPY_SAMPLES = int(1_000_000)  # max number of voxels to use for histogram when approximating
BINS = 256  # histogram bins for entropy calculation
CSV_PATH = 'meta_df_entropy.csv'
COREGISTERED_FOLDER = 'df_coregistered_02_09_2026_V6.6/raw_studies'  # Folder name to replace 'raw_studies' with
from tqdm import tqdm
tqdm.pandas()

# Try to import ANTsPy
try:
    import ants
    ANTS_AVAILABLE = True
except ImportError:
    ANTS_AVAILABLE = False
    print("Warning: ANTsPy not available. antspy_spacing column will contain NaN values.")

def _get_voxel_dims_from_row(row):
    """
    Extract voxel dimensions and unit from row.
    Returns: (dims_tuple, unit_string) or (None, None)
    """
    key_sets = [
        ('voxel_x','voxel_y','voxel_z'),
        ('spacing_x','spacing_y','spacing_z'),
        ('vx','vy','vz'),
        ('voxel_size_x','voxel_size_y','voxel_size_z'),
        ('voxel_size',),
        ('spacing',),
        ('voxel_dims',),
        ('pixel_size',),
    ]
    
    # Try to find unit information
    unit = None
    for unit_col in ['voxel_unit', 'spacing_unit', 'unit', 'voxel_units', 'spacing_units']:
        if unit_col in row.index and pd.notna(row[unit_col]):
            unit = str(row[unit_col]).strip()
            break
    
    for keys in key_sets:
        if all(k in row.index for k in keys):
            vals = row[list(keys)]
            if len(keys) == 1:
                v = vals.iloc[0]
                if v is None:
                    continue
                if hasattr(v, '__iter__') and not isinstance(v, str):
                    arr = np.asarray(v, dtype=float)
                    if arr.size >= 3:
                        return tuple(arr[:3].astype(float)), unit
                    elif arr.size == 2:
                        return (float(arr[0]), float(arr[1]), 1.0), unit
                    else:
                        continue
            else:
                try:
                    return tuple(map(float, vals.values[:3])), unit
                except Exception:
                    continue
    return None, None

def _get_voxel_dims_from_file(path):
    """
    Extract voxel dimensions and unit from file headers.
    Returns: (dims_tuple, unit_string) or (None, None)
    """
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.nii', '.nii.gz') and nib is not None:
        try:
            img = nib.load(path)
            zooms = img.header.get_zooms()
            # Try to get unit from NIfTI header
            unit = 'mm'  # Default for NIfTI
            try:
                # NIfTI xyzt_units field
                xyzt_units = img.header.get_xyzt_units()
                if xyzt_units[0]:
                    unit = xyzt_units[0]
            except Exception:
                pass
            
            if len(zooms) >= 3:
                return tuple(map(float, zooms[:3])), unit
            elif len(zooms) == 2:
                return (float(zooms[0]), float(zooms[1]), 1.0), unit
        except Exception:
            pass
    if tifffile is not None and ext in ('.tif', '.tiff'):
        try:
            with tifffile.TiffFile(path) as tif:
                unit = 'pixels'  # Default for TIFF
                ij_metadata = tif.imagej_metadata
                if ij_metadata:
                    # Try to get unit from ImageJ metadata
                    if 'unit' in ij_metadata:
                        unit = str(ij_metadata['unit'])
                    spacing = ij_metadata.get('spacing')
                    if spacing:
                        try:
                            sp = float(spacing)
                            return (1.0, 1.0, sp), unit
                        except Exception:
                            pass
                for page in tif.pages:
                    tags = page.tags
                    if 'XResolution' in tags and 'YResolution' in tags:
                        try:
                            xr = tags['XResolution'].value
                            yr = tags['YResolution'].value
                            def rational_to_float(r):
                                if isinstance(r, tuple) and len(r) == 2:
                                    return r[0] / r[1] if r[1] != 0 else 1.0
                                return float(r)
                            # Check for ResolutionUnit tag
                            if 'ResolutionUnit' in tags:
                                res_unit = tags['ResolutionUnit'].value
                                # 1 = None, 2 = inches, 3 = centimeters
                                if res_unit == 2:
                                    unit = 'inches'
                                elif res_unit == 3:
                                    unit = 'cm'
                            return (rational_to_float(xr), rational_to_float(yr), 1.0), unit
                        except Exception:
                            pass
        except Exception:
            pass
    if sitk is not None:
        try:
            img = sitk.ReadImage(path)
            spacing = img.GetSpacing()
            unit = 'mm'  # Default for medical images
            if len(spacing) >= 3:
                return tuple(map(float, spacing[:3])), unit
            elif len(spacing) == 2:
                return (float(spacing[0]), float(spacing[1]), 1.0), unit
        except Exception:
            pass
    return None, None

def _get_antspy_spacing(path):
    """
    Extract spacing from image using ANTsPy.
    Returns: tuple of spacing values or None if error/not available
    """
    if not ANTS_AVAILABLE:
        return None
    
    if not isinstance(path, str) or not os.path.exists(path):
        return None
    
    try:
        # Read image with ANTsPy
        img = ants.image_read(path)
        # Get spacing - returns tuple
        spacing = img.spacing
        return tuple(float(s) for s in spacing)
    except Exception:
        return None

def _compute_entropy_from_array(arr, bins=BINS, approximate=APPROXIMATE_ENTROPY, max_samples=MAX_ENTROPY_SAMPLES):
    try:
        flat = arr.ravel()
        n = flat.size
        if n == 0:
            return 0.0
        # Subsample deterministically to avoid large memory/cpu cost
        if approximate and n > max_samples:
            step = int(np.ceil(n / max_samples))
            sample = flat[::step]
        else:
            sample = flat
        # handle NaNs by masking
        if np.issubdtype(sample.dtype, np.floating):
            # ignore NaNs in float data
            mask = ~np.isnan(sample)
            if not np.any(mask):
                return 0.0
            sample = sample[mask]
            mn = float(np.min(sample))
            mx = float(np.max(sample))
            if np.isclose(mx, mn):
                return 0.0
            counts, _ = np.histogram(sample, bins=bins, range=(mn, mx))
        else:
            try:
                mn = int(sample.min())
                mx = int(sample.max())
            except Exception:
                mn = int(np.nanmin(sample))
                mx = int(np.nanmax(sample))
            if np.isclose(mx, mn):
                return 0.0
            if mx - mn + 1 <= bins:
                counts, _ = np.histogram(sample, bins=(mx - mn + 1), range=(mn - 0.5, mx + 0.5))
            else:
                counts, _ = np.histogram(sample, bins=bins, range=(mn, mx))
        probs = counts.astype(np.float64)
        s = probs.sum()
        if s == 0:
            return 0.0
        probs /= s
        nz = probs > 0
        entropy = -np.sum(probs[nz] * np.log2(probs[nz]))
        return float(entropy)
    except Exception:
        return float('nan')

def _read_array(path):
    ext = os.path.splitext(path)[1].lower()
    if tifffile is not None and ext in ('.tif', '.tiff'):
        try:
            with tifffile.TiffFile(path) as tif:
                return tif.asarray(memmap=True)
        except Exception:
            try:
                return tifffile.imread(path)
            except Exception:
                pass
    if nib is not None and ext in ('.nii', '.nii.gz'):
        try:
            img = nib.load(path)
            return img.get_fdata(dtype=np.float32)
        except Exception:
            pass
    if sitk is not None:
        try:
            img = sitk.ReadImage(path)
            arr = sitk.GetArrayFromImage(img)
            return arr
        except Exception:
            pass
    if ext == '.npy':
        return np.load(path, mmap_mode='r')
    raise RuntimeError(f"Could not read image {path}; unsupported format or missing libraries.")

def _get_file_size_mb(path):
    """Get file size in megabytes"""
    try:
        if os.path.exists(path):
            size_bytes = os.path.getsize(path)
            return size_bytes / (1024 * 1024)  # Convert to MB
        return np.nan
    except Exception:
        return np.nan

def _get_volume_unit(voxel_unit):
    """
    Convert linear unit to cubic unit.
    E.g., 'mm' -> 'mm³', 'um' -> 'um³', 'pixels' -> 'pixels³'
    """
    if not voxel_unit or pd.isna(voxel_unit):
        return None
    
    unit_str = str(voxel_unit).strip()
    
    # Handle common variations
    unit_map = {
        'mm': 'mm³',
        'millimeter': 'mm³',
        'millimeters': 'mm³',
        'um': 'um³',
        'μm': 'μm³',
        'micrometer': 'um³',
        'micrometers': 'um³',
        'nm': 'nm³',
        'nanometer': 'nm³',
        'nanometers': 'nm³',
        'cm': 'cm³',
        'centimeter': 'cm³',
        'centimeters': 'cm³',
        'm': 'm³',
        'meter': 'm³',
        'meters': 'm³',
        'pixel': 'pixels³',
        'pixels': 'pixels³',
        'px': 'px³',
        'inch': 'in³',
        'inches': 'in³',
        'in': 'in³',
    }
    
    # Check if unit is in our map
    unit_lower = unit_str.lower()
    if unit_lower in unit_map:
        return unit_map[unit_lower]
    
    # If not in map, just add ³ to the end
    return f"{unit_str}³"

def _compute_segmentation_volume(seg_path, voxel_volume):
    """
    Compute the volume of a segmentation by counting non-zero voxels
    and multiplying by voxel_volume.
    
    Args:
        seg_path: Path to segmentation file
        voxel_volume: Volume of a single voxel (product of spacing dimensions)
        
    Returns:
        Volume in cubic units (same as voxel_volume units), or NaN if error
    """
    try:
        if not isinstance(seg_path, str) or not os.path.exists(seg_path):
            return np.nan
        
        if np.isnan(voxel_volume):
            return np.nan
            
        # Read the segmentation array
        seg_array = _read_array(seg_path)
        
        # Count non-zero voxels
        non_zero_count = np.count_nonzero(seg_array)
        
        # Calculate volume
        volume = non_zero_count * voxel_volume
        
        return float(volume)
        
    except Exception as e:
        return np.nan

def _find_coregistered_files(row, coregistered_folder='df_coregistered2_05_2026_V6.4.1/raw_studies'):
    """
    Finds coregistered files by replacing 'raw_studies' in the image_path and segmentation_path
    with the specified coregistered folder name.
    Returns (coregistered_image_path, coregistered_segmentation_path) tuple.
    
    Args:
        row: DataFrame row containing 'image path' and 'segmentation path' columns
        coregistered_folder: Folder name to replace 'raw_studies' with
    """
    try:
        coreg_image = np.nan
        coreg_seg = np.nan
        
        # Get image path from row
        for col in ['image path', 'Image Path', 'image_path']:
            if col in row.index and pd.notna(row[col]):
                img_path = str(row[col])
                print(f"Original image path: {img_path}")
                
                # Replace 'raw_studies' with coregistered folder name (case-insensitive)
                if 'raw_studies' in img_path.lower():
                    # Find the actual case-sensitive occurrence
                    idx = img_path.lower().find('raw_studies')
                    coreg_image = img_path[:idx] + coregistered_folder + img_path[idx+len('raw_studies'):]
                    coreg_image = coreg_image.replace('.nii.gz','_coregistered.nii.gz')
                    print(f"Checking coregistered image path: {coreg_image}")
                    
                    # Check if file exists
                    if not os.path.exists(coreg_image):
                        print(f"  → File NOT found")
                        coreg_image = np.nan
                    else:
                        print(f"  → File FOUND")
                        pass  # FIXED: Changed 'break' to 'pass'
                break
        
        # Get segmentation path from row
        for col in ['segmentation path', 'Segmentation Path', 'segmentation_path']:
            if col in row.index and pd.notna(row[col]):
                seg_path = str(row[col])
                print(f"Original segmentation path: {seg_path}")
                
                # Replace 'raw_studies' with coregistered folder name (case-insensitive)
                if 'raw_studies' in seg_path.lower():
                    # Find the actual case-sensitive occurrence
                    idx = seg_path.lower().find('raw_studies')
                    coreg_seg = seg_path[:idx] + coregistered_folder + seg_path[idx+len('raw_studies'):]
                    coreg_seg = coreg_seg.replace('.nii.gz','_coregistered.nii.gz')
                    #print(f"Checking coregistered segmentation path: {coreg_seg}")
                    
                    # Check if file exists
                    if not os.path.exists(coreg_seg):
                        #print(f"  → File NOT found")
                        coreg_seg = np.nan
                    else:
                        #print(f"  → File FOUND")
                        pass  # This was already correct
                    break  # FIXED: Moved 'break' to correct indentation level
        
        print(f"Returning: ({coreg_image}, {coreg_seg})\n")
        return coreg_image, coreg_seg
        
    except Exception as e:
        print(f"Exception occurred: {e}\n")
        return np.nan, np.nan

def _process_path(path, row):
    if not isinstance(path, str) or not os.path.exists(path):
        return pd.Series({
            'entropy': np.nan, 
            'voxel_volume': np.nan,
            'voxel_unit': None,
            'volume_unit': None,
            'file_size_mb': np.nan,
            'coregistered_image': np.nan,
            'coregistered_segmentation': np.nan,
            'segmentation_volume': np.nan,
            'antspy_spacing': None
        })
    
    # Get file size
    file_size_mb = _get_file_size_mb(path)
    
    # Find coregistered files (both image and segmentation)
    coreg_image, coreg_seg = _find_coregistered_files(row, COREGISTERED_FOLDER)
    
    # Get ANTsPy spacing
    antspy_spacing = _get_antspy_spacing(path)
    
    # Get voxel dimensions and unit
    dims, voxel_unit = _get_voxel_dims_from_row(row)
    if dims is None:
        dims, voxel_unit = _get_voxel_dims_from_file(path)
    
    voxel_volume = np.nan
    volume_unit = None
    
    if dims is not None:
        try:
            voxel_volume = float(np.prod(dims))
            volume_unit = _get_volume_unit(voxel_unit)
        except Exception:
            voxel_volume = np.nan
    
    # Compute entropy for the image
    try:
        arr = _read_array(path)
        entropy = _compute_entropy_from_array(arr)
    except Exception:
        entropy = float('nan')
    
    # Compute segmentation volume using the existing 'segmentation path' column
    segmentation_volume = np.nan
    for seg_col in ['segmentation path', 'Segmentation Path', 'segmentation_path']:
        if seg_col in row.index and pd.notna(row[seg_col]):
            seg_path = row[seg_col]
            segmentation_volume = _compute_segmentation_volume(seg_path, voxel_volume)
            break
    
    return pd.Series({
        'entropy': entropy, 
        'voxel_volume': voxel_volume,
        'voxel_unit': voxel_unit,
        'volume_unit': volume_unit,
        'file_size_mb': file_size_mb,
        'coregistered_image': coreg_image,
        'coregistered_segmentation': coreg_seg,
        'segmentation_volume': segmentation_volume,
        'antspy_spacing': antspy_spacing
    })

def _run_compute_and_save(meta):
    _meta = meta.copy()
    results = _meta.progress_apply(lambda r: _process_path(r['image path'], r), axis=1)
    _meta[['entropy', 'voxel_volume', 'voxel_unit', 'volume_unit', 'file_size_mb', 'coregistered_image', 'coregistered_segmentation', 'segmentation_volume', 'antspy_spacing']] = results
    _meta.to_csv(CSV_PATH, index=False)
    print(f"Saved results to {CSV_PATH}")
    return _meta

# If meta_df_entropy.csv exists, load it and skip computation. Otherwise compute and save.
if os.path.exists(CSV_PATH):
    try:
        meta_df = pd.read_csv(CSV_PATH)
        print(f"Loaded existing {CSV_PATH}; skipping computation.")
    except Exception as e:
        if 'meta_df' not in globals():
            raise RuntimeError(f"Failed to read {CSV_PATH}: {e}\nNo in-memory meta_df available.")
        print(f"Error reading {CSV_PATH}: {e}. Computing from in-memory meta_df...")
        meta_df = _run_compute_and_save(meta_df)
else:
    if 'meta_df' not in globals():
        raise RuntimeError("meta_df not found in notebook and no meta_df_entropy.csv present.")
    print(f"Computing metrics for {len(meta_df)} rows...")
    meta_df = _run_compute_and_save(meta_df)

Computing metrics for 1793 rows...


  0%|          | 0/1793 [00:00<?, ?it/s]

Original image path: ./raw_studies/t2_thick/126_0_0/126_0_0_t2_thick_image.nii.gz
Checking coregistered image path: ./df_coregistered_02_09_2026_V6.6/raw_studies/t2_thick/126_0_0/126_0_0_t2_thick_image_coregistered.nii.gz
  → File NOT found
Original segmentation path: ./raw_studies/t2_thick/126_0_0/126_0_0_t2_thick_RVS _uvauser2.nii.gz
Returning: (nan, nan)



  0%|          | 2/1793 [00:00<09:51,  3.03it/s]

Original image path: ./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1+_thick_cor_image.nii.gz
Checking coregistered image path: ./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1+_thick_cor_image_coregistered.nii.gz
  → File FOUND
Original segmentation path: ./raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1+_thick_cor_RVS _uvauser2.nii.gz
Returning: (./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1+_thick_cor_image_coregistered.nii.gz, ./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_cor/126_0_0/126_0_0_t1+_thick_cor_RVS _uvauser2_coregistered.nii.gz)



...

Original image path: ./raw_studies/t2_thin/95_2_574/95_2_574_t2_thin_image.nii.gz
Checking coregistered image path: ./df_coregistered_02_09_2026_V6.6/raw_studies/t2_thin/95_2_574/95_2_574_t2_thin_image_coregistered.nii.gz
  → File FOUND
Original segmentation path: ./raw_studies/t2_thin/95_2_574/95_2_574_t2_thin_RVS _uvauser2-copy.nii.gz
Returning: (./df_coregistered_02_09_2026_V6.6/raw_studies/t2_thin/95_2_574/95_2_574_t2_thin_image_coregistered.nii.gz, ./df_coregistered_02_09_2026_V6.6/raw_studies/t2_thin/95_2_574/95_2_574_t2_thin_RVS _uvauser2-copy_coregistered.nii.gz)



100%|██████████| 1793/1793 [15:29<00:00,  3.13it/s]

Original image path: ./raw_studies/t1+_thick_cor/95_2_574/95_2_574_t1+_thick_cor_image.nii.gz
Checking coregistered image path: ./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_cor/95_2_574/95_2_574_t1+_thick_cor_image_coregistered.nii.gz
  → File FOUND
Original segmentation path: ./raw_studies/t1+_thick_cor/95_2_574/95_2_574_t1+_thick_cor_RVS _uvauser2.nii.gz
Returning: (./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_cor/95_2_574/95_2_574_t1+_thick_cor_image_coregistered.nii.gz, ./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_cor/95_2_574/95_2_574_t1+_thick_cor_RVS _uvauser2_coregistered.nii.gz)

Original image path: ./raw_studies/t1+_thick_ax/95_2_574/95_2_574_t1+_thick_ax_image.nii.gz
Checking coregistered image path: ./df_coregistered_02_09_2026_V6.6/raw_studies/t1+_thick_ax/95_2_574/95_2_574_t1+_thick_ax_image_coregistered.nii.gz
  → File FOUND
Original segmentation path: ./raw_studies/t1+_thick_ax/95_2_574/95_2_574_t1+_thick_ax_RVS _uvauser2.nii.gz
Returni

100%|██████████| 1793/1793 [15:29<00:00,  1.93it/s]

Saved results to meta_df_entropy.csv


In [23]:
# for meta_df find the assicated image in acceptable and see if there is a file that is coregistered or seg_coregistered 

In [24]:
meta_df['image path'].iloc[0]

'./raw_studies/t2_thick/126_0_0/126_0_0_t2_thick_image.nii.gz'

In [25]:
# Create boolean columns for each modality (study type)
# First for any scan of that modality
modality_presence = meta_df.groupby('Patient_MRI_Days Tracker')['Study Type'].apply(
    lambda x: pd.Series(True, index=x.unique())
).unstack(fill_value=False).reset_index()

# Add prefix to column names for clarity
modality_presence.columns = ['Patient_MRI_Days Tracker'] + [f'has_{col}' for col in modality_presence.columns[1:]]

# Now for scans with segmentations
modality_with_seg = meta_df[meta_df['original_seg_path'].notna()].groupby('Patient_MRI_Days Tracker')['Study Type'].apply(
    lambda x: pd.Series(True, index=x.unique())
).unstack(fill_value=False).reset_index()

# Add prefix to column names
modality_with_seg.columns = ['Patient_MRI_Days Tracker'] + [f'has_seg_{col}' for col in modality_with_seg.columns[1:]]

# Merge both dataframes
tracker_modality_df = modality_presence.merge(modality_with_seg, on='Patient_MRI_Days Tracker', how='left')

# Fill NaN values with False using infer_objects
tracker_modality_df = tracker_modality_df.fillna(False).infer_objects(copy=False)

print("Patient Timepoints with Modality Presence and Segmentation Status:")
display(tracker_modality_df.head(20))

Patient Timepoints with Modality Presence and Segmentation Status:


/tmp/ipykernel_431655/1039564943.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tracker_modality_df = tracker_modality_df.fillna(False).infer_objects(copy=False)


,Patient_MRI_Days Tracker,has_t1+_thick_ax,has_t1+_thick_cor,has_t1+_thin,has_t2_thick,has_t2_thin,has_seg_t1+_thick_ax,has_seg_t1+_thick_cor,has_seg_t1+_thin,has_seg_t2_thick,has_seg_t2_thin
0,126_0_0,True,True,False,True,False,True,True,False,True,False
1,126_1_312,True,True,False,False,True,True,True,False,False,True
2,126_2_582,True,True,False,False,True,True,True,False,False,True
3,126_3_921,True,True,False,False,True,True,True,False,False,True
4,126_4_1231,True,True,False,False,True,True,True,False,False,True
5,126_5_1607,True,True,False,False,True,True,True,False,False,True
6,131_0_0,True,True,False,False,True,True,False,False,False,True
7,131_1_196,False,False,True,False,True,False,False,True,False,True
8,131_2_258,False,False,True,False,True,False,False,True,False,True
9,132_0_0,True,True,False,False,True,False,True,False,False,True


In [26]:
# Filter for rows that have BOTH has_seg_t1_thin AND has_seg_t2_thin
trackers_with_both_segs = tracker_modality_df[
    (tracker_modality_df['has_seg_t1+_thin'] == True) & 
    (tracker_modality_df['has_seg_t2_thin'] == True)
]

print(f"Trackers with BOTH t1_thin AND t2_thin segmentations: {len(trackers_with_both_segs)}")
display(trackers_with_both_segs)

Trackers with BOTH t1_thin AND t2_thin segmentations: 82


,Patient_MRI_Days Tracker,has_t1+_thick_ax,has_t1+_thick_cor,has_t1+_thin,has_t2_thick,has_t2_thin,has_seg_t1+_thick_ax,has_seg_t1+_thick_cor,has_seg_t1+_thin,has_seg_t2_thick,has_seg_t2_thin
7,131_1_196,False,False,True,False,True,False,False,True,False,True
8,131_2_258,False,False,True,False,True,False,False,True,False,True
11,132_2_232,False,False,True,False,True,False,False,True,False,True
17,140_2_780,False,True,True,False,True,False,True,True,False,True
21,147_0_0,False,True,True,False,True,False,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...
590,745_2_589,True,True,True,True,True,True,True,True,True,True
591,745_3_960,True,True,True,False,True,True,True,True,False,True
592,747_1_6,True,True,True,False,True,True,True,True,False,True
609,94_0_0,False,False,True,False,True,False,False,True,False,True


In [27]:
#compares T1+_thin and T2_thin of the same time point with dicescore
import pandas as pd
import numpy as np
import nibabel as nib
from scipy.spatial.distance import dice

def calculate_dice_coefficient(seg1_path, seg2_path):
    """
    Calculate Dice coefficient between two segmentation masks.
    
    Parameters:
    seg1_path: path to first segmentation file
    seg2_path: path to second segmentation file
    
    Returns:
    dice_score: Dice coefficient (0-1, where 1 is perfect overlap)
    """
    try:
        # Load segmentation files
        seg1 = nib.load(seg1_path).get_fdata()
        seg2 = nib.load(seg2_path).get_fdata()
        
        # Binarize segmentations (in case they're not already binary)
        seg1_binary = (seg1 > 0).astype(np.uint8)
        seg2_binary = (seg2 > 0).astype(np.uint8)
        
        # Calculate Dice coefficient
        # Dice = 2 * |A ∩ B| / (|A| + |B|)
        intersection = np.sum(seg1_binary * seg2_binary)
        sum_volumes = np.sum(seg1_binary) + np.sum(seg2_binary)
        
        if sum_volumes == 0:
            return np.nan  # Both segmentations are empty
        
        dice_score = 2.0 * intersection / sum_volumes
        return dice_score
        
    except Exception as e:
        print(f"Error calculating Dice for {seg1_path} and {seg2_path}: {e}")
        return np.nan


def create_dice_comparison_dataframe(df):
    """
    Create a dataframe comparing T1 and T2 segmentations with Dice scores.
    
    Parameters:
    df: pandas DataFrame with imaging data
    
    Returns:
    result_df: DataFrame with columns [Patient_MRI_Days Tracker, t1_thin, t2_thin, dice_score]
    """
    results = []
    
    # Group by Patient_MRI_Days Tracker
    grouped = df.groupby('Patient_MRI_Days Tracker')
    
    for tracker_id, group in grouped:
        # Identify T1 and T2 scans based on Series Description or Study Type
        # Adjust the column name and values based on your data
        t1_scans = group[group['Study Type'].str.contains('t1', case=False, na=False)]
        t2_scans = group[group['Study Type'].str.contains('t2', case=False, na=False)]
        
        # If you have multiple T1 or T2 scans per tracker, iterate through combinations
        for _, t1_row in t1_scans.iterrows():
            for _, t2_row in t2_scans.iterrows():
                # Check if both have coregistered segmentations
                if pd.notna(t1_row['coregistered_segmentation']) and \
                   pd.notna(t2_row['coregistered_segmentation']):
                    
                    # Calculate Dice score
                    dice_score = calculate_dice_coefficient(
                        t1_row['coregistered_segmentation'],
                        t2_row['coregistered_segmentation']
                    )
                    
                    # Store results
                    results.append({
                        'Patient_MRI_Days Tracker': tracker_id,
                        't1_thin': t1_row['coregistered_image'],
                        't2_thin': t2_row['coregistered_image'],
                        't1_segmentation': t1_row['coregistered_segmentation'],
                        't2_segmentation': t2_row['coregistered_segmentation'],
                        'dice_score': dice_score
                    })
    
    # Create result dataframe
    result_df = pd.DataFrame(results)
    
    return result_df


# Usage example:
dice_df = create_dice_comparison_dataframe(meta_df)
display(dice_df.head())

# Count how many Patient_MRI_Days Trackers have both T1 and T2 with segmentations
count = dice_df['Patient_MRI_Days Tracker'].nunique()
print(f"Number of patient timepoints with both T1 and T2 segmentations: {count}")

# Or get the total number of comparisons made
total_comparisons = len(dice_df)
print(f"Total T1-T2 comparisons: {total_comparisons}")

# If you want to see which ones they are:
print("\nPatient_MRI_Days Trackers with both T1 and T2:")
print(dice_df['Patient_MRI_Days Tracker'].unique())

,Patient_MRI_Days Tracker,t1_thin,t2_thin,t1_segmentation,t2_segmentation,dice_score
0,126_1_312,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.083344
1,126_1_312,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.374806
2,126_1_312,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.374806
3,126_2_582,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.776452
4,126_2_582,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.808989


Number of patient timepoints with both T1 and T2 segmentations: 266
Total T1-T2 comparisons: 487

Patient_MRI_Days Trackers with both T1 and T2:
['126_1_312' '126_2_582' '126_3_921' '126_5_1607' '131_0_0' '131_1_196'
 '131_2_258' '132_0_0' '132_2_232' '137_0_0' '140_0_0' '140_2_780'
 '146_0_0' '146_2_941' '147_1_104' '148_0_0' '148_1_182' '148_2_253'
 '152_1_95' '152_3_905' '152_4_1291' '152_5_1683' '169_0_0' '169_1_184'
 '169_2_722' '169_3_835' '170_1_176' '170_2_382' '170_3_732' '178_0_0'
 '178_5_2486' '185_1_277' '188_0_0' '188_1_263' '188_2_645' '188_3_1033'
 '188_4_1434' '190_1_150' '193_0_0' '193_1_175' '199_0_0' '199_1_183'
 '207_0_0' '207_3_1194' '210_2_231' '215_2_322' '218_0_0' '218_1_199'
 '222_1_11' '222_2_402' '229_1_212' '229_3_938' '235_0_0' '235_3_552'
 '258_2_96' '262_1_497' '267_1_195' '267_2_554' '267_3_916' '267_4_1278'
 '267_5_1693' '270_0_0' '270_1_324' '286_0_0' '301_0_0' '301_1_229'
 '301_2_614' '304_0_0' '304_1_206' '304_2_305' '308_1_189' '308_2_521'
 '309_1_1

In [28]:
from tqdm import tqdm

#compare the pairwise dice with each other in a timepoint
def create_pairwise_dice_comparison(df):
    """
    Create pairwise Dice score comparisons for all studies within each timepoint.
    
    Parameters:
    df: pandas DataFrame with imaging data
    
    Returns:
    result_df: DataFrame with pairwise comparisons and Dice scores
    """
    results = []
    
    # Group by Patient_ID
    grouped = df.groupby('Patient_ID')
    
    # Count total comparisons for progress bar
    total_comparisons = 0
    for _, group in grouped:
        studies_with_seg = group[pd.notna(group['coregistered_segmentation'])]
        n_studies = len(studies_with_seg)
        if n_studies >= 2:
            total_comparisons += n_studies * (n_studies - 1) // 2
    
    print(f"Processing {len(grouped)} patients with {total_comparisons} total pairwise comparisons...")
    
    # Create progress bar
    pbar = tqdm(total=total_comparisons, desc="Calculating Dice scores", unit="comparison")
    
    for tracker_id, group in grouped:
        # Get all studies with segmentations at this timepoint
        studies_with_seg = group[pd.notna(group['coregistered_segmentation'])].copy()
        
        # Create all pairwise combinations
        n_studies = len(studies_with_seg)
        
        if n_studies < 2:
            continue  # Skip if less than 2 studies to compare
        
        # Iterate through all pairs
        for i in range(n_studies):
            for j in range(i + 1, n_studies):
                study1 = studies_with_seg.iloc[i]
                study2 = studies_with_seg.iloc[j]
                
                # Calculate Dice score
                dice_score = calculate_dice_coefficient(
                    study1['coregistered_segmentation'],
                    study2['coregistered_segmentation']
                )
                
                # Store results
                results.append({
                    'Patient_MRI_Days Tracker': tracker_id,
                    'study1_filename': study1['coregistered_image'],
                    'study2_filename': study2['coregistered_image'],
                    'study1_type': study1.get('Study Type', 'Unknown'),
                    'study2_type': study2.get('Study Type', 'Unknown'),
                    'study1_segmentation': study1['coregistered_segmentation'],
                    'study2_segmentation': study2['coregistered_segmentation'],
                    'dice_score': dice_score
                })
                
                # Update progress bar
                pbar.update(1)
    
    # Close progress bar
    pbar.close()
    
    # Create result dataframe
    result_df = pd.DataFrame(results)
    
    return result_df

# Create pairwise comparison dataframe
pairwise_dice_df = create_pairwise_dice_comparison(meta_df)

# Display results
display(pairwise_dice_df.head(10))

# Summary statistics
print(f"\nTotal pairwise comparisons: {len(pairwise_dice_df)}")
print(f"Number of unique timepoints with comparisons: {pairwise_dice_df['Patient_MRI_Days Tracker'].nunique()}")
print(f"\nDice score statistics:")
print(pairwise_dice_df['dice_score'].describe())

# Show distribution of comparisons per timepoint
comparisons_per_timepoint = pairwise_dice_df.groupby('Patient_MRI_Days Tracker').size()
print(f"\nComparisons per timepoint:")
print(f"Mean: {comparisons_per_timepoint.mean():.2f}")
print(f"Max: {comparisons_per_timepoint.max()}")
print(f"Min: {comparisons_per_timepoint.min()}")

Processing 205 patients with 3690 total pairwise comparisons...


Calculating Dice scores: 100%|██████████| 3690/3690 [45:18<00:00,  1.36comparison/s]


,Patient_MRI_Days Tracker,study1_filename,study2_filename,study1_type,study2_type,study1_segmentation,study2_segmentation,dice_score
0,50,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_cor,t2_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.747266
1,50,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_cor,t1+_thick_ax,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.705104
2,50,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,t1+_thick_ax,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.831360
3,53,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.000000
4,57,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.680782
5,58,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,t1+_thick_cor,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.483980
6,58,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,t2_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.586031
7,58,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,t1+_thick_cor,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.000000
8,58,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.000000
9,58,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,t1+_thick_ax,./df_coregistered_02_09_2026_V6.6/raw_studies/...,./df_coregistered_02_09_2026_V6.6/raw_studies/...,0.000000



Total pairwise comparisons: 3690
Number of unique timepoints with comparisons: 175

Dice score statistics:
count    3608.000000
mean        0.455007
std         0.286232
min         0.000000
25%         0.206410
50%         0.512785
75%         0.693599
max         1.000000
Name: dice_score, dtype: float64

Comparisons per timepoint:
Mean: 21.09
Max: 136
Min: 1


In [29]:
pairwise_dice_df.to_csv('pairwise_dice_df.csv')

In [30]:
def create_average_dice_per_study(pairwise_df):
    """
    Calculate average Dice score for each study based on all comparisons it participated in.
    
    Parameters:
    pairwise_df: DataFrame with pairwise comparisons (output from create_pairwise_dice_comparison)
    
    Returns:
    avg_dice_df: DataFrame with average Dice score per study
    """
    results = []
    
    # Get all unique studies from both columns
    all_studies = pd.concat([
        pairwise_df[['Patient_MRI_Days Tracker', 'study1_filename', 'study1_type', 'study1_segmentation']].rename(
            columns={'study1_filename': 'filename', 'study1_type': 'study_type', 'study1_segmentation': 'segmentation'}
        ),
        pairwise_df[['Patient_MRI_Days Tracker', 'study2_filename', 'study2_type', 'study2_segmentation']].rename(
            columns={'study2_filename': 'filename', 'study2_type': 'study_type', 'study2_segmentation': 'segmentation'}
        )
    ]).drop_duplicates()
    
    print(f"Calculating average Dice scores for {len(all_studies)} unique studies...")
    
    # For each unique study, find all comparisons it participated in
    for _, study_row in tqdm(all_studies.iterrows(), total=len(all_studies), desc="Processing studies"):
        tracker_id = study_row['Patient_MRI_Days Tracker']
        filename = study_row['filename']
        study_type = study_row['study_type']
        segmentation = study_row['segmentation']
        
        # Find all comparisons involving this study
        comparisons_as_study1 = pairwise_df[
            (pairwise_df['Patient_MRI_Days Tracker'] == tracker_id) & 
            (pairwise_df['study1_filename'] == filename)
        ]['dice_score']
        
        comparisons_as_study2 = pairwise_df[
            (pairwise_df['Patient_MRI_Days Tracker'] == tracker_id) & 
            (pairwise_df['study2_filename'] == filename)
        ]['dice_score']
        
        # Combine all Dice scores for this study
        all_dice_scores = pd.concat([comparisons_as_study1, comparisons_as_study2])
        
        # Remove NaN values
        all_dice_scores = all_dice_scores.dropna()
        
        if len(all_dice_scores) > 0:
            results.append({
                'Patient_MRI_Days Tracker': tracker_id,
                'study_filename': filename,
                'study_type': study_type,
                'segmentation': segmentation,
                'n_comparisons': len(all_dice_scores),
                'mean_dice': all_dice_scores.mean(),
                'std_dice': all_dice_scores.std(),
                'min_dice': all_dice_scores.min(),
                'max_dice': all_dice_scores.max(),
                'median_dice': all_dice_scores.median()
            })
    
    # Create result dataframe
    avg_dice_df = pd.DataFrame(results)
    
    # Sort by mean_dice descending
    avg_dice_df = avg_dice_df.sort_values('mean_dice', ascending=False).reset_index(drop=True)
    
    return avg_dice_df

# Create average Dice score dataframe
avg_dice_per_study_df = create_average_dice_per_study(pairwise_dice_df)

# Display results
print("\n" + "="*70)
print("Average Dice Scores Per Study")
print("="*70)
display(avg_dice_per_study_df.head(20))

# Summary statistics
print("\n" + "="*70)
print("Overall Statistics:")
print("="*70)
print(f"Total studies analyzed: {len(avg_dice_per_study_df)}")
print(f"\nMean Dice statistics across all studies:")
print(avg_dice_per_study_df['mean_dice'].describe())

print(f"\nNumber of comparisons per study:")
print(avg_dice_per_study_df['n_comparisons'].describe())

# Group by study type
print("\n" + "="*70)
print("Average Dice by Study Type:")
print("="*70)
study_type_summary = avg_dice_per_study_df.groupby('study_type').agg({
    'mean_dice': ['mean', 'std', 'count'],
    'n_comparisons': 'mean'
}).round(4)
print(study_type_summary)

# Show best and worst performing studies
print("\n" + "="*70)
print("Top 10 Studies (Highest Average Dice):")
print("="*70)
display(avg_dice_per_study_df.head(10)[['Patient_MRI_Days Tracker', 'study_filename', 'study_type', 'n_comparisons', 'mean_dice']])

print("\n" + "="*70)
print("Bottom 10 Studies (Lowest Average Dice):")
print("="*70)
display(avg_dice_per_study_df.tail(10)[['Patient_MRI_Days Tracker', 'study_filename', 'study_type', 'n_comparisons', 'mean_dice']])
avg_dice_per_study_df.to_csv('avg_dice_per_study_df.csv', index=False)

Calculating average Dice scores for 1072 unique studies...


Processing studies: 100%|██████████| 1072/1072 [00:01<00:00, 747.37it/s]


Average Dice Scores Per Study


,Patient_MRI_Days Tracker,study_filename,study_type,segmentation,n_comparisons,mean_dice,std_dice,min_dice,max_dice,median_dice
0,315,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,1,0.898512,NaN,0.898512,0.898512,0.898512
1,315,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,1,0.898512,NaN,0.898512,0.898512,0.898512
2,569,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,./df_coregistered_02_09_2026_V6.6/raw_studies/...,6,0.874854,0.097701,0.798655,1.000000,0.825907
3,525,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,6,0.860297,0.021891,0.832031,0.885778,0.859867
4,185,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,2,0.859517,0.040423,0.830933,0.888100,0.859517
5,553,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,4,0.859260,0.014307,0.839108,0.872353,0.862790
6,525,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,6,0.855925,0.032157,0.814766,0.892438,0.863065
7,525,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_cor,./df_coregistered_02_09_2026_V6.6/raw_studies/...,6,0.855726,0.038097,0.801787,0.892438,0.867979
8,553,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,./df_coregistered_02_09_2026_V6.6/raw_studies/...,4,0.852151,0.026173,0.825106,0.876693,0.853403
9,553,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_cor,./df_coregistered_02_09_2026_V6.6/raw_studies/...,4,0.851935,0.022336,0.823908,0.876693,0.853569



Overall Statistics:
Total studies analyzed: 1070

Mean Dice statistics across all studies:
count    1070.000000
mean        0.474142
std         0.225682
min         0.000000
25%         0.339669
50%         0.501707
75%         0.638032
max         0.898512
Name: mean_dice, dtype: float64

Number of comparisons per study:
count    1070.000000
mean        6.981308
std         4.156394
min         1.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        36.000000
Name: n_comparisons, dtype: float64

Average Dice by Study Type:
              mean_dice               n_comparisons
                   mean     std count          mean
study_type                                         
t1+_thick_ax     0.4802  0.2124   282        7.5355
t1+_thick_cor    0.4703  0.2163   292        7.3562
t1+_thin         0.4719  0.2472   200        5.5850
t2_thin          0.4736  0.2328   296        7.0270

Top 10 Studies (Highest Average Dice):


,Patient_MRI_Days Tracker,study_filename,study_type,n_comparisons,mean_dice
0,315,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,1,0.898512
1,315,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,1,0.898512
2,569,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_ax,6,0.874854
3,525,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,6,0.860297
4,185,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,2,0.859517
5,553,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,4,0.859260
6,525,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,6,0.855925
7,525,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_cor,6,0.855726
8,553,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,4,0.852151
9,553,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thick_cor,4,0.851935



Bottom 10 Studies (Lowest Average Dice):


,Patient_MRI_Days Tracker,study_filename,study_type,n_comparisons,mean_dice
1060,601,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,5,0.0
1061,735,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,2,0.0
1062,94,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t2_thin,6,0.0
1063,515,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,2,0.0
1064,495,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,3,0.0
1065,493,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,2,0.0
1066,483,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,2,0.0
1067,439,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,8,0.0
1068,434,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,1,0.0
1069,421,./df_coregistered_02_09_2026_V6.6/raw_studies/...,t1+_thin,8,0.0
